# [미션2] DUR 금기 의약품 목록 분류 데이터 분석

## 0. 준비

**데이터 업로드 방법**: 병용금기 파일이 255MB로 크기 때문에 `files.upload()`는 매우 느립니다.
구글 드라이브에 `07_DUR_Data` 폴더를 만들어 CSV 5개를 올린 뒤 아래처럼 마운트하는 방법을 권장합니다.

In [ ]:
# 코랩에 없는 라이브러리 설치
!pip install konlpy gensim

In [ ]:
# 구글 드라이브 마운트 (드라이브 내 07_DUR_Data 폴더에 CSV 5개를 올려둔 경우)
from google.colab import drive
drive.mount('/content/drive')

PATH = '/content/drive/MyDrive/07_DUR_Data/'     # 폴더 경로 (끝에 / 포함)

import os
print(os.listdir(PATH))

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# CSV 파일이 cp949(한글 윈도우) 인코딩이라 encoding='cp949' 필수
# (생략하면 UnicodeDecodeError 발생)
def read_dur(name):
    return pd.read_csv(PATH + f'의약품안전사용서비스(DUR)_{name} 품목리스트 2026.6.csv',
                       encoding='cp949')

임부 = read_dur('임부금기')
연령 = read_dur('연령금기')
노인 = read_dur('노인주의')
노인해열 = read_dur('노인주의(해열진통소염제)')

print('임부금기:', 임부.shape, list(임부.columns))
임부.head(3)

In [ ]:
# 병용금기는 87만 행(255MB)이라 필요한 컬럼만 읽어서 메모리 절약
병용 = pd.read_csv(PATH + '의약품안전사용서비스(DUR)_병용금기 품목리스트 2026.6.csv',
                  encoding='cp949',
                  usecols=['성분명A', '성분코드A', '제품코드A', '성분명B', '성분코드B', '제품코드B',
                           '고시번호', '고시일자', '상세정보', '비고'])
print('병용금기:', 병용.shape)
병용.head(3)

### 전처리: 제품 단위 → 성분(조합) 단위로 집계
- 원본 파일은 **제품 1개 = 1행**(같은 성분의 제품이 수백 개씩 반복)
- 변수 정의서의 분석 단위는 **성분(조합) 1개 = 1행**이고, 여기에 **제품수**(해당 금기에 걸리는 제품 수) 컬럼이 붙음
- 따라서 성분(조합)별로 행 수를 세어 `제품수`를 만들고, 중복 행을 제거

In [ ]:
# groupby(...).transform('size') : 같은 그룹의 행이 몇 개인지를 각 행에 붙여줌
# drop_duplicates() : 그룹마다 첫 행만 남김 → 성분(조합) 단위 데이터 완성

# (1) 병용금기: 성분코드A + 성분코드B 조합 단위
병용['제품수'] = 병용.groupby(['성분코드A', '성분코드B'])['제품코드A'].transform('size')
병용_성분 = 병용.drop_duplicates(['성분코드A', '성분코드B']).copy()
병용_성분 = 병용_성분.rename(columns={'성분코드A': '성분코드', '성분명A': '성분명',
                                  '성분코드B': '병용성분코드', '성분명B': '병용성분명'})
병용_성분['금기유형'] = '병용금기'
print('병용금기 조합 수:', len(병용_성분), '/ 제품수 최대:', 병용_성분['제품수'].max())

# (2) 임부금기: 성분코드 단위
임부['제품수'] = 임부.groupby('성분코드')['제품코드'].transform('size')
임부_성분 = 임부.drop_duplicates('성분코드').copy()
임부_성분['금기유형'] = '임부금기'
print('임부금기 성분 수:', len(임부_성분), '/ 제품수 최대:', 임부_성분['제품수'].max())

In [ ]:
# (3) 연령금기 / 노인주의도 같은 방식으로 정리
연령['제품수'] = 연령.groupby(['성분코드', '특정연령', '연령처리조건'])['제품코드'].transform('size')
연령_성분 = 연령.drop_duplicates(['성분코드', '특정연령', '연령처리조건']).copy()
연령_성분['금기유형'] = '연령금기'

# 노인주의 파일은 컬럼명이 달라서 통일 (약품상세정보 → 상세정보, 공고일자 → 고시일자)
노인 = 노인.rename(columns={'약품상세정보': '상세정보', '공고일자': '고시일자', '공고번호': '고시번호'})
노인['제품수'] = 노인.groupby('성분코드')['제품코드'].transform('size')
노인_성분 = 노인.drop_duplicates('성분코드').copy()
노인_성분['금기유형'] = '노인주의'

노인해열 = 노인해열.rename(columns={'약품상세정보': '상세정보'})
노인해열['제품수'] = 노인해열.groupby('성분코드')['제품코드'].transform('size')
노인해열_성분 = 노인해열.drop_duplicates('성분코드').copy()
노인해열_성분['금기유형'] = '노인주의_해열진통소염제'

print(len(연령_성분), len(노인_성분), len(노인해열_성분))

In [ ]:
# (4) 5개 금기유형을 하나로 합치기 (변수 정의서의 통합 테이블)
사용컬럼 = ['금기유형', '성분코드', '성분명', '병용성분코드', '병용성분명', '금기등급',
          '특정연령', '특정연령단위', '연령처리조건', '상세정보', '비고', '고시번호', '고시일자', '제품수']

dur = pd.concat([병용_성분, 임부_성분, 연령_성분, 노인_성분, 노인해열_성분], ignore_index=True)
dur = dur.reindex(columns=사용컬럼)      # 없는 컬럼은 자동으로 결측(NaN) 처리

print(dur.shape)
print(dur['금기유형'].value_counts())
dur.head(3)

In [ ]:
# 형태소 분석기 준비 (상세정보 텍스트 분석용)
from konlpy.tag import Okt
okt = Okt()

print(okt.nouns('임부 투여금기, 동물실험에서 간독성의 증거 나타남'))

---
## 1. 병용금기 중 제품수 합계 상위 100개 성분의 상세정보 최빈 단어

In [ ]:
# 병용금기만 뽑아서 성분명별 제품수 합계 → 상위 100개
병용금기 = dur[dur['금기유형'] == '병용금기']
top100 = 병용금기.groupby('성분명')['제품수'].sum().sort_values(ascending=False).head(100)

print('상위 100개 성분의 제품수 합계 범위:', top100.min(), '~', top100.max())
top100.head(10)

In [ ]:
# 상위 100개 성분에 해당하는 행들의 상세정보(금기 사유) 모으기
top100_data = 병용금기[병용금기['성분명'].isin(top100.index)]
print('행 수:', len(top100_data), '/ 상세정보 종류:', top100_data['상세정보'].nunique())

# Okt로 명사 추출 (2글자 이상만)
nouns_list = []
for text in top100_data['상세정보'].dropna():
    nouns_list.extend([n for n in okt.nouns(str(text)) if len(n) > 1])

word_counts = pd.Series(nouns_list).value_counts()
print('=== 상세정보 최빈 단어 상위 15 ===')
display(word_counts.head(15))

In [ ]:
# 같은 성분 조합이 수천 건씩 반복되면 같은 문장이 그만큼 여러 번 세어짐
# → 문장(상세정보) 중복을 제거하고 다시 세어 비교
unique_text = top100_data['상세정보'].dropna().drop_duplicates()
print('중복 제거 후 문장 수:', len(unique_text))

nouns_unique = []
for text in unique_text:
    nouns_unique.extend([n for n in okt.nouns(str(text)) if len(n) > 1])

print('=== 중복 제거 기준 상위 15 ===')
display(pd.Series(nouns_unique).value_counts().head(15))

### 1번 결과
- 병용금기에서 제품수 합계 상위 100개 성분: 1위 **metformin hydrochloride(159,731)**, 2위 iohexol(129,438), 3위 itraconazole(101,084). 상위 100개의 제품수 합계는 410~159,731이다.
- 이 성분들의 상세정보는 20,410행이지만, **문장 종류는 217개뿐**이다. 같은 금기 사유가 수천 번 반복된다.
- **행 기준 최빈 단어**: 촉진(14,914) · 신부전(14,383) · 기능(14,151) · 유산(14,088) · 산성(14,088)
  - 모두 `기능적 신부전에 의해 유산 산성증 촉진`이라는 한 문장에서 나온다. metformin + 조영제(iohexol 등) 조합이 워낙 많아 이 문장이 1만 번 넘게 반복된 결과다.
- **문장 중복을 제거한 최빈 단어**: 증가(103) · 농도(83) · 감소(67) · 혈중(65) · 투여(59)
  - 중복을 빼면 병용금기 사유의 공통 뼈대가 드러난다. 대부분 **'혈중 농도 증가/감소'**에 관한 내용이다.
- **→ 병용금기 사유는 한 약물이 다른 약물의 혈중 농도를 바꿔 위험을 키우는 내용이 핵심이다.** 단, 가장 많이 걸리는 조합은 메트포르민-조영제의 유산산증(젖산산증) 위험이다.

---
## 2. 고시 경과기간 계산 + 임부금기 등급(1/2)에 따른 차이 검정

In [ ]:
임부금기 = dur[dur['금기유형'] == '임부금기'].copy()

# 고시일자 → 날짜형으로 변환 후 경과기간(연) 계산
임부금기['고시일자'] = pd.to_datetime(임부금기['고시일자'], errors='coerce')
오늘 = pd.Timestamp.today()
임부금기['경과기간'] = (오늘 - 임부금기['고시일자']).dt.days / 365.25   # 365.25 = 윤년 포함 평균

print('고시일자 범위:', 임부금기['고시일자'].min().date(), '~', 임부금기['고시일자'].max().date())
print('금기등급 분포:'); print(임부금기['금기등급'].value_counts(dropna=False))

In [ ]:
# 등급이 1 또는 2인 데이터만 사용 (M 등급 40건, 결측 1건은 제외)
임부금기['금기등급'] = 임부금기['금기등급'].astype(str)
d2 = 임부금기[임부금기['금기등급'].isin(['1', '2'])].dropna(subset=['경과기간'])
print('검정 데이터:', len(d2))

d2.groupby('금기등급')['경과기간'].describe().round(2)

In [ ]:
from scipy import stats

grade1 = d2[d2['금기등급'] == '1']['경과기간']
grade2 = d2[d2['금기등급'] == '2']['경과기간']

# 정규성 검정 (p < 0.05 이면 정규분포가 아님)
print('정규성 - 1등급:', stats.shapiro(grade1).pvalue)
print('정규성 - 2등급:', stats.shapiro(grade2).pvalue)

In [ ]:
# 정규분포가 아니므로 비모수 검정(Mann-Whitney U) 사용
# 귀무가설: 1등급과 2등급의 고시 경과기간에 차이가 없다
print('1등급 중앙값:', round(grade1.median(), 2), '/ 2등급 중앙값:', round(grade2.median(), 2))
print(stats.mannwhitneyu(grade1, grade2))

# (참고) 평균 비교
print('1등급 평균:', round(grade1.mean(), 2), '/ 2등급 평균:', round(grade2.mean(), 2))
print(stats.ttest_ind(grade1, grade2, equal_var=False))

### 2번 결과
- 고시일자 범위는 2016-12-30 ~ 2025-12-23이고, 경과기간은 오늘 기준으로 계산했다.

| 등급 | 건수 | 평균 | 중앙값 | 1사분위 |
|---|---|---|---|---|
| 1등급 | 638 | 9.01년 | 9.73년 | 9.73년 |
| 2등급 | 3,332 | 7.72년 | 9.73년 | 5.74년 |

- 정규성 검정: 1등급 p=1.3e-38, 2등급 p=1.7e-55 → 정규분포가 아니므로 Mann-Whitney U 검정 사용
- **Mann-Whitney U: p = 4.8e-38 < 0.05 → 귀무가설 기각.** t-검정(p=3.9e-55)도 같은 결론이다.
- 주의해서 읽어야 할 점: **중앙값은 두 집단 모두 9.73년으로 같다.** 2016-12-30에 한꺼번에 고시된 건이 많아 그 값에 절반 이상이 몰려 있기 때문이다. 차이는 평균(9.01년 vs 7.72년)과 1사분위(9.73년 vs 5.74년)에서 나타난다. 즉 **2등급에는 최근에 고시된 건이 많고, 1등급은 대부분 오래된 고시**다.
- **→ 임부금기 등급에 따라 고시 경과기간 분포에 유의미한 차이가 있다. 1등급이 2등급보다 오래 전에 고시되었다.**

---
## 3. 투여경로(A/B/C)와 임부금기 등급의 관계 검정 + 경로별 1등급 비율
- 성분코드 7번째 글자 = 투여경로 (A 내복 / B 주사 / C 외용 / D 기타)

In [ ]:
# 문자열에서 7번째 글자 = 인덱스 6 (0부터 시작)
임부금기['투여경로'] = 임부금기['성분코드'].str[6]
print(임부금기['투여경로'].value_counts())

# 예시 확인
임부금기[['성분코드', '투여경로']].head()

In [ ]:
# 등급 1·2, 투여경로 A·B·C만 사용
d3 = 임부금기[임부금기['금기등급'].isin(['1', '2']) & 임부금기['투여경로'].isin(['A', 'B', 'C'])]

route_table = pd.crosstab(d3['투여경로'], d3['금기등급'])
route_table['1등급비율'] = (route_table['1'] / (route_table['1'] + route_table['2'])).round(3)
route_table

In [ ]:
# 카이제곱 독립성 검정
# 귀무가설: 투여경로와 금기등급은 관계가 없다(독립이다)
chi2, p, dof, expected = stats.chi2_contingency(route_table[['1', '2']])
print('카이제곱 통계량:', round(chi2, 3))
print('p-value:', p)
print('자유도:', dof)
print('기대빈도 최솟값:', round(expected.min(), 1))     # 5 이상이면 검정 조건 만족

### 3번 결과
| 투여경로 | 1등급 | 2등급 | 1등급 비율 |
|---|---|---|---|
| A (내복) | 518 | 2,104 | **19.8%** |
| B (주사) | 97 | 863 | 10.1% |
| C (외용) | 23 | 365 | 5.9% |

- 카이제곱 검정: **χ² = 81.333, df = 2, p = 2.2e-18 < 0.05 → 귀무가설 기각** (기대빈도 최솟값 62.4로 검정 조건 만족)
- **→ 투여경로와 임부금기 등급은 관계가 있다.** 내복약의 1등급 비율(19.8%)이 외용약(5.9%)의 3배가 넘는다.
- 해석: 먹는 약은 전신에 흡수되어 태아에게 도달하기 쉬운 반면, 바르는 약은 전신 흡수가 적어 위험도가 낮게 분류된 것으로 보인다.
- 참고: 주사(B)가 내복(A)보다 1등급 비율이 낮은데, 임신부에게 일상적으로 처방되는 약이 주로 내복약이라 절대 금기 지정의 실익이 큰 쪽도 내복약이기 때문으로 추정된다. 이 데이터만으로는 확인할 수 없다.

### 3-1. (보충) 등급 × 투여경로별 사유 문구 정리
투여경로에 따라 위험도가 다르다면 **금기 사유 문구도 다르게 쓰이는지** 확인한다. 6·7번 모델 해석의 근거가 된다.

In [ ]:
# 결측을 '(결측)'으로 표시해 함께 집계
d3 = d3.copy()
d3['상세정보'] = d3['상세정보'].fillna('(결측)')

# 등급 × 경로 건수와, 셀마다 문구가 몇 종류인지
print(pd.crosstab(d3['투여경로'], d3['금기등급'], margins=True))
print()
print(d3.groupby(['금기등급', '투여경로'])['상세정보'].nunique())

In [ ]:
# 셀별 대표 문구 (상위 3개)
for g in ['1', '2']:
    for r in ['A', 'B', 'C']:
        top = d3[(d3['금기등급'] == g) & (d3['투여경로'] == r)]['상세정보'].value_counts().head(3)
        print(f'=== {g}등급 / {r} (n={len(d3[(d3.금기등급==g)&(d3.투여경로==r)])})')
        for text, cnt in top.items():
            print(f'  {cnt:4d}  {text[:70]}')
        print()

In [ ]:
# 같은 문구가 1등급과 2등급에 함께 쓰이는지 확인
#  nunique()가 2 = 그 문구가 1등급·2등급 양쪽에 모두 등장
공유 = d3.groupby('상세정보')['금기등급'].nunique()
공유문구 = 공유[공유 > 1].index
print('1·2등급에 함께 쓰인 문구:', len(공유문구), '종류 /', d3['상세정보'].isin(공유문구).sum(), '건')

표 = pd.crosstab(d3[d3['상세정보'].isin(공유문구)]['상세정보'], d3['금기등급'])
표['합계'] = 표['1'] + 표['2']
표.sort_values('합계', ascending=False).head(8)

### 3-1 결과 해석
**경로별 문구 성격이 다르다**
- A 내복: 성분 고유의 구체적 위험 (`임신초기 3개월간 고용량 투여시 최기형 가능성` 69건 — 비타민A)
- B 주사: 방사성의약품·조영제의 방사선 위험 (`태아 갑상선기능저하증 유발 가능 / 방사선 절차는 태아 기형 유발 가능` 99건)
- C 외용: 제형 조건과 **사용량 제한** (`임부에게 대량 또는 장기간에 걸쳐 광범위하게 사용하지 말 것` 24건, `(외용, 점안제...)` 19건) → "쓰지 마라"가 아니라 **"적게, 짧게 써라"**가 핵심

**문제 1. 1등급 주사제는 사유가 비어 있는 경우가 가장 많다**
- 1등급 B 97건 중 결측 23건 + `-` 13건 = **36건(37%)**. 가장 위험한 등급인데 사유가 없다 → 우선 보완 대상

**문제 2. 같은 문구가 등급을 가르지 못한다 (11종, 772건 = 전체의 19.4%)**

| 문구 | 2등급 | 1등급 |
|---|---|---|
| 임부에 대한 안전성 미확립. | 263 | 16 |
| 임부에 대한 안전성 미확립 (마침표 없음) | 184 | 1 |
| (결측) | 141 | 29 |
| `-` | 38 | 19 |
| **임부 투여금기** | **19** | 10 |

- `임부 투여금기`는 1등급 문구처럼 보이지만 오히려 2등급에 더 많다.
- 즉 문구는 등급을 정의하는 규칙이 아니라 **참고 설명**이다. 6번 모델이 정확도 0.958을 내는 것은 문구가 등급을 그대로 알려주기 때문이 아니라, **위험 서술의 강도**(최기형성·기형 유발 vs 안전성 미확립)를 학습했기 때문으로 보는 것이 타당하다.

**문제 3. 마침표 하나로 문구가 갈라진다**
- `임부에 대한 안전성 미확립.`(279건)과 `임부에 대한 안전성 미확립`(185건)은 같은 내용인데 시스템에서는 서로 다른 문구다. 합치면 464건으로 전체의 12%.

---
## 4. 임부금기 상세정보 Word2Vec → '태아'와 유사한 단어

In [ ]:
from gensim.models import Word2Vec

# 상세정보 한 건을 문장 하나로 보고 명사 리스트 만들기
sentences = []
for text in 임부금기['상세정보'].dropna():
    sentences.append([n for n in okt.nouns(str(text)) if len(n) > 1])

print('문장 수:', len(sentences))
print("'태아' 등장 횟수:", sum(s.count('태아') for s in sentences))
print(sentences[:3])

In [ ]:
# Skip-Gram (sg=1): 중심 단어로 주변 단어를 예측
model_sg = Word2Vec(sentences, vector_size=100, window=3, min_count=2,
                    sg=1, epochs=100, seed=42, workers=1)

print("=== Skip-Gram: '태아'와 유사한 단어 ===")
for word, score in model_sg.wv.most_similar('태아', topn=10):
    print(word, round(score, 3))

In [ ]:
# CBOW (sg=0): 주변 단어로 중심 단어를 예측 → 자주 나오는 단어에 유리
model_cbow = Word2Vec(sentences, vector_size=100, window=3, min_count=2,
                      sg=0, epochs=100, seed=42, workers=1)

print("=== CBOW: '태아'와 유사한 단어 ===")
for word, score in model_cbow.wv.most_similar('태아', topn=10):
    print(word, round(score, 3))

In [ ]:
# 유사어가 실제로 어떤 문장에 쓰였는지 확인
for word in ['태자', '모체']:
    예시 = 임부금기[임부금기['상세정보'].fillna('').str.contains(word)]['상세정보']
    print(f'[{word}] {len(예시)}건')
    print(예시.head(2).tolist())

### 4번 결과
- 임부금기 상세정보 3,840건에서 '태아'는 **2,389번** 등장한다(1번 데이터와 달리 충분히 많다).
- **CBOW 결과(더 적절)**: 보고(0.41) · **태자**(0.40) · 가능성(0.40) · 유발(0.37) · 나타남(0.36) · 중기(0.31) · **모체**(0.29)
  - **태자**(동물실험에서 쓰는 태아의 표현, 674건), **모체**(334건)는 '태아'와 실제로 같은 맥락에서 쓰이는 단어다. 예: `동물실험에서 고용량 투여시 모체에 대한 독성에 의해 태자미숙 및 저체중의 발생빈도 증가 보고`
  - 유발·보고·가능성은 `태아 기형 유발 가능`, `태아 독성 보고` 같은 서술 표현이다.
- **Skip-Gram 결과**: 억제제·현저·핍뇨·골절 등으로 유사도가 0.3대에 머물고 해석이 어렵다.
- **→ '태아'와 가장 유사한 단어는 태자, 모체이며, 함께 쓰이는 서술어는 유발·보고·가능성이다.** 이번 데이터는 문장이 3,840건으로 많고 '태아'도 자주 나와서, 자주 나오는 단어에 강한 **CBOW가 더 안정적인 결과**를 보였다.

---
## 5. 제품수 합계가 가장 많은 성분 → 그 성분이 어떤 병용성분과 제품수가 가장 많은지

In [ ]:
# 전체 금기유형을 합쳐 성분명별 제품수 합계
성분별 = dur.groupby('성분명')['제품수'].sum().sort_values(ascending=False)
성분별.head(10)

In [ ]:
top_ingredient = 성분별.index[0]
print('제품수 합계가 가장 많은 성분:', top_ingredient, '/ 합계:', 성분별.iloc[0])

# 그 성분의 병용금기 상대 성분별 제품수
상대성분 = (dur[(dur['성분명'] == top_ingredient) & (dur['금기유형'] == '병용금기')]
          .groupby('병용성분명')['제품수'].sum().sort_values(ascending=False))
상대성분.head(10)

In [ ]:
# 1위 조합의 금기 사유 확인
best = 상대성분.index[0]
print(f'{top_ingredient} + {best} : 제품수 {상대성분.iloc[0]}')
print(dur[(dur['성분명'] == top_ingredient) & (dur['병용성분명'] == best)]['상세정보'].values[:2])

### 5번 결과
- 제품수 합계가 가장 많은 성분: **metformin hydrochloride(메트포르민, 합계 159,731)**. 당뇨병 1차 치료제로 제품 수가 매우 많다.
- 이 성분이 가장 많은 제품수를 기록한 병용 조합:

| 병용성분 | 제품수 |
|---|---|
| **iohexol(이오헥솔)** | **86,404** |
| iopamidol | 38,085 |
| iomeprol | 8,855 |
| iodixanol | 7,195 |

- 상위 조합이 모두 **요오드 조영제**이고, 금기 사유는 `기능적 신부전에 의해 유산 산성증 촉진`으로 동일하다.
- **→ 메트포르민 + 조영제 조합이 DUR에서 가장 광범위한 병용금기다.** 조영제 사용 시 신기능이 떨어지면 메트포르민이 몸에 쌓여 젖산산증(유산산증)을 일으킬 수 있기 때문이다.

---
## 6. 상세정보 + 투여경로 + 제품수 + 경과기간 → 임부금기 1등급 판별 (Scikit-learn)
- 상세정보(텍스트) → **TF-IDF**
- 투여경로(범주) → **원-핫 인코딩**
- 제품수, 경과기간(숫자) → 그대로 사용

In [ ]:
# 목표 변수: 1등급이면 1, 2등급이면 0
model_df = 임부금기[임부금기['금기등급'].isin(['1', '2'])].dropna(subset=['경과기간']).copy()
model_df['1등급'] = (model_df['금기등급'] == '1').astype(int)
print('데이터:', len(model_df), '/ 1등급 비율:', round(model_df['1등급'].mean(), 3))

# 상세정보 → 명사만 공백으로 이은 문장 (TF-IDF 입력용)
model_df['상세정보_명사'] = [' '.join([n for n in okt.nouns(str(t)) if len(n) > 1])
                        for t in model_df['상세정보'].fillna('')]
model_df[['상세정보', '상세정보_명사']].head(3)

In [ ]:
# 투여경로 원-핫 인코딩 + 숫자형 변수 붙이기
etc = pd.get_dummies(model_df['투여경로'], prefix='경로').astype(int)
etc['제품수'] = model_df['제품수']
etc['경과기간'] = model_df['경과기간']
etc.head(3)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 학습 80% / 평가 20% (stratify: 1등급 비율 유지)
train_idx, test_idx = train_test_split(model_df.index, test_size=0.2,
                                       stratify=model_df['1등급'], random_state=42)

# TF-IDF는 학습 데이터로만 fit
tfidf = TfidfVectorizer(max_features=300)
train_text = tfidf.fit_transform(model_df.loc[train_idx, '상세정보_명사']).toarray()
test_text = tfidf.transform(model_df.loc[test_idx, '상세정보_명사']).toarray()

# 텍스트 + 나머지 변수 합치기
X_train = np.hstack([train_text, etc.loc[train_idx].values])
X_test = np.hstack([test_text, etc.loc[test_idx].values])
y_train = model_df.loc[train_idx, '1등급']
y_test = model_df.loc[test_idx, '1등급']
print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# class_weight='balanced' : 1등급이 16%로 적으므로 가중치 보정
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

print('정확도:', round(accuracy_score(y_test, pred), 3))
print('(비교) 전부 2등급으로 찍었을 때:', round(1 - y_test.mean(), 3))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=['2등급', '1등급'], digits=3))

In [ ]:
# 변수 중요도
feature_names = list(tfidf.get_feature_names_out()) + list(etc.columns)
importance = pd.Series(rf.feature_importances_, index=feature_names)
importance.sort_values(ascending=False).head(10)

In [ ]:
# 텍스트 없이 정형 변수(투여경로·제품수·경과기간)만으로 학습하면?
n_etc = etc.shape[1]
rf2 = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf2.fit(X_train[:, -n_etc:], y_train)
pred2 = rf2.predict(X_test[:, -n_etc:])

print('정형 변수만 정확도:', round(accuracy_score(y_test, pred2), 3))
print(classification_report(y_test, pred2, target_names=['2등급', '1등급'], digits=3))

### 6번 결과
- 데이터 3,970건(1등급 16.1%), 입력: 상세정보 TF-IDF(300개 단어) + 투여경로 원-핫(3) + 제품수 + 경과기간
- 랜덤포레스트 결과: **정확도 0.958** (전부 2등급으로 찍으면 0.839)
  - 1등급 재현율 **0.898**, 정밀도 0.852 → 소수인 1등급도 잘 잡아낸다.
- 정형 변수(투여경로·제품수·경과기간)만 쓰면 정확도가 **0.567**로 떨어진다.
- **→ 판별력의 대부분은 상세정보 텍스트에서 나온다.** 실제로 1등급은 `임부 투여금기`, 2등급은 `임부에 대한 안전성 미확립` 같은 표현이 반복되어, 사유 문장이 등급을 사실상 규정한다.
- 앞선 미션의 대출 데이터와 달리 **상세정보는 고시 시점에 함께 작성되는 정보라 데이터 누수가 아니다.** 새로 고시되는 금기의 사유 문장으로 등급을 자동 분류하는 보조 도구로 쓸 수 있다.

---
## 7. 상세정보 → 임부금기 1등급 판별 RNN 모델 (model_dur.h5 저장)
순서: 텍스트 → 정수 시퀀스(Text to Sequence) → 패딩(Padding) → Embedding + SimpleRNN

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 6번에서 나눈 학습/평가 데이터를 그대로 사용
train_texts = model_df.loc[train_idx, '상세정보_명사']
test_texts = model_df.loc[test_idx, '상세정보_명사']

# (1) Tokenizer: 단어마다 번호 부여 (학습 데이터 기준)
tokenizer = Tokenizer(oov_token='OOV')     # 학습에 없던 단어는 OOV 처리
tokenizer.fit_on_texts(train_texts)
vocab_size = len(tokenizer.word_index) + 1
print('단어 수:', vocab_size)

# (2) Text to Sequence
train_seq = tokenizer.texts_to_sequences(train_texts)
test_seq = tokenizer.texts_to_sequences(test_texts)
print(train_texts.iloc[0], '→', train_seq[0])

In [ ]:
# (3) Padding: 길이를 똑같이 맞추기
#     상세정보는 길이 차이가 커서 95% 분위수로 자름 (너무 긴 문장 1~2건 때문에 전체가 길어지는 것 방지)
lengths = [len(s) for s in train_seq]
print('단어 수 평균:', round(np.mean(lengths), 1), '/ 최대:', max(lengths))

max_len = int(np.percentile(lengths, 95))
print('maxlen:', max_len)

X_train_pad = pad_sequences(train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(test_seq, maxlen=max_len, padding='post', truncating='post')
print(X_train_pad[0])

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
import tensorflow as tf

tf.random.set_seed(42)

# RNN 계열 알고리즘 사용 (SimpleRNN)
model = Sequential()
model.add(Embedding(vocab_size, 64))       # 단어 번호 → 64차원 벡터
model.add(SimpleRNN(32))                   # 단어를 순서대로 읽으며 문장 특징 추출
model.add(Dropout(0.5))                    # 과적합 방지
model.add(Dense(1, activation='sigmoid'))  # 1등급일 확률

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# 1등급(16%)이 적으므로 가중치 보정 후 학습
class_weight = {0: 1, 1: 5}

history = model.fit(X_train_pad, y_train, epochs=10, batch_size=32,
                    validation_split=0.2, class_weight=class_weight)

In [ ]:
# 평가
loss, acc = model.evaluate(X_test_pad, y_test)
print('평가 정확도:', round(acc, 3))
print('(비교) 전부 2등급으로 찍었을 때:', round(1 - y_test.mean(), 3))

pred_rnn = (model.predict(X_test_pad) > 0.5).astype(int).ravel()
print(confusion_matrix(y_test, pred_rnn))
print(classification_report(y_test, pred_rnn, target_names=['2등급', '1등급'], digits=3))

In [ ]:
# 학습 과정 그래프
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.show()

In [ ]:
# 모델 저장 (.h5)
model.save('model_dur.h5')

# 저장한 모델 불러와 새 문장으로 예측
from tensorflow.keras.models import load_model
loaded_model = load_model('model_dur.h5')

new_text = '임부 투여금기, 동물실험에서 기형 유발이 보고됨'
new_seq = tokenizer.texts_to_sequences([' '.join([n for n in okt.nouns(new_text) if len(n) > 1])])
new_pad = pad_sequences(new_seq, maxlen=max_len, padding='post')
print(new_text, '→ 1등급 확률:', round(float(loaded_model.predict(new_pad)[0][0]), 3))

In [ ]:
# 파일 다운로드
from google.colab import files
files.download('model_dur.h5')

### 7번 결과
- 상세정보 → Tokenizer(단어 1,137개) → 정수 시퀀스 → 패딩(길이 49, 95% 분위) → Embedding(64) + **SimpleRNN(32)** 신경망을 구성하고 **model_dur.h5**로 저장했다.
- 평가 정확도 **0.884** (전부 2등급으로 찍으면 0.839), 1등급 재현율 **0.891**, 정밀도 0.594
- 1등급을 놓치는 경우는 128건 중 14건뿐이다. 다만 2등급 78건을 1등급으로 잘못 분류해 정밀도는 낮다. **안전 관리 목적에서는 놓치는 것(위양성보다 위음성)이 더 위험하므로 재현율이 높은 편이 바람직하다.**
- 저장한 모델로 새 문장 `임부 투여금기, 동물실험에서 기형 유발이 보고됨`을 예측하면 1등급 확률 **0.94**가 나온다.
- 6번 랜덤포레스트(0.958)보다는 낮다. 문장이 짧고(평균 17단어) 표현이 정형화되어 있어, 단어 순서를 학습하는 RNN보다 단어 존재 여부를 보는 TF-IDF 방식이 더 잘 맞는 것으로 보인다.
- (신경망은 실행할 때마다 결과가 조금씩 달라질 수 있다.)

---
## 8. 병용금기 상세정보(중복 제거) 토픽 모델링 (LDA)
- LDA(Latent Dirichlet Allocation): 문서를 여러 '주제(토픽)'의 혼합으로 보고, 주제별 주요 단어를 찾아내는 방법

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 병용금기 상세정보에서 중복 문장 제거
docs_raw = 병용금기['상세정보'].dropna().drop_duplicates()
print('중복 제거 후 문서 수:', len(docs_raw))

# 명사만 남기기
docs = [' '.join([n for n in okt.nouns(str(t)) if len(n) > 1]) for t in docs_raw]
print(docs[:3])

In [ ]:
# LDA 입력은 단어 빈도(BoW) 행렬
# max_df=0.6 : 60% 넘는 문서에 나오는 흔한 단어 제외
# min_df=2   : 1개 문서에만 나오는 단어 제외
cv = CountVectorizer(max_df=0.6, min_df=2)
X_lda = cv.fit_transform(docs)
print('문서-단어 행렬:', X_lda.shape)

In [ ]:
# 토픽 개수를 3~6개로 바꿔가며 perplexity(낮을수록 좋음) 비교
for k in [3, 4, 5, 6]:
    lda_tmp = LatentDirichletAllocation(n_components=k, random_state=42, max_iter=30).fit(X_lda)
    print(f'토픽 {k}개 → perplexity {round(lda_tmp.perplexity(X_lda), 1)}')

In [ ]:
# 토픽 5개로 최종 학습
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=30)
lda.fit(X_lda)

words = cv.get_feature_names_out()
for i, topic in enumerate(lda.components_):
    top_words = [words[j] for j in topic.argsort()[-10:][::-1]]   # 비중 높은 단어 10개
    print(f'토픽 {i+1}:', ', '.join(top_words))

In [ ]:
# 각 토픽에 속하는 문장 예시 확인 (해석용)
topic_result = lda.transform(X_lda)          # 문서마다 토픽별 비중
docs_raw = docs_raw.reset_index(drop=True)

for i in range(5):
    print(f'--- 토픽 {i+1} 대표 문장')
    for idx in topic_result[:, i].argsort()[-2:][::-1]:
        print('  ', docs_raw[idx][:70])

### 8번 결과
- 병용금기 상세정보를 중복 제거하면 **365개 문장**이고, 단어 211개로 문서-단어 행렬을 만들었다.
- 토픽 개수별 perplexity(낮을수록 좋음): 3개 108.2 / 4개 107.5 / **5개 106.9** / 6개 110.9 → **5개 토픽** 선택

| 토픽 | 주요 단어 | 해석 |
|---|---|---|
| 1 | 감소, 효과, 농도, 혈장, 치료 | **약효 감소형** — 병용 시 혈중 농도가 떨어져 치료 효과가 사라짐 (예: 소포스부비르 치료 효과 감소) |
| 2 | 횡문근, 병증, 독성, 경련, 근육, 혈관 | **근육·혈관 독성형** — 횡문근융해증, 맥각독성 |
| 3 | 투여, 반응, 이상, 종료, 금기, 약물 | **이상반응·투여금기형** — 병용투여 금기 및 투여 종료 후 기간 안내 |
| 4 | 증후군, 정신, 독성, 세로토닌, 변화 | **증후군형** — 세로토닌 증후군, 종양용해 증후군 등 |
| 5 | 상승, 혈압, 연장, 고혈압, 심실, 부정맥 | **심혈관형** — 혈압 상승, QT 연장, 부정맥 |

- **→ 병용금기 사유는 크게 ① 약효가 사라지는 위험, ② 농도가 올라가 생기는 독성(근육·심혈관·증후군), ③ 투여 금기 안내의 세 갈래로 구성된다.** 위험의 성격이 다르므로 알림 문구나 경고 강도를 토픽별로 다르게 설계할 수 있다.

---
## 종합 정리
| 질문 | 발견 |
|---|---|
| 가장 광범위한 금기 조합 | **메트포르민 + 요오드 조영제**(iohexol 86,404건). 사유는 신기능 저하에 의한 유산산증 촉진 |
| 고시 경과기간과 등급 | 유의미한 차이 있음(p=4.8e-38). 1등급 평균 9.01년 vs 2등급 7.72년 → **최근 고시는 2등급 비중이 높다** |
| 투여경로와 등급 | 유의미한 차이 있음(p=2.2e-18). 1등급 비율 **내복 19.8% > 주사 10.1% > 외용 5.9%** |
| 금기 사유 텍스트 | '태아'는 태자·모체와 유사. 병용금기 사유는 약효 감소 / 근육·심혈관 독성 / 증후군 / 투여금기 안내의 5개 토픽으로 구성 |
| 등급 판별 모델 | 상세정보 TF-IDF + 정형 변수 랜덤포레스트 **정확도 0.958**, RNN(SimpleRNN) **0.884** |

**N기관에 대한 시사점**
1. 상세정보(금기 사유) 텍스트만으로 임부금기 1등급을 90% 가까운 재현율로 가려낼 수 있다. 신규 고시 성분의 **등급 초안 자동 제안**에 활용할 수 있다.
2. 정형 변수(투여경로·제품수·경과기간)만으로는 정확도 0.567에 그친다. **텍스트 분석이 반드시 병행되어야 한다.**
3. 내복약의 1등급 비율이 외용약의 3배가 넘으므로, 투여경로별로 **경고 강도를 차등 적용**하는 것이 타당하다.
4. 병용금기 사유가 5개 토픽으로 나뉘므로, DUR 팝업 문구를 토픽별(약효 감소 / 독성 / 심혈관 / 증후군 / 투여금기)로 표준화하면 현장에서 위험의 성격을 빠르게 파악할 수 있다.

---
## 비즈니스 관점 해석 (N기관 관점)

DUR의 핵심 가치는 **처방·조제 현장에서 뜨는 경고창**이다. 경고가 정확하면 사고를 막고, 부정확하거나 너무 많으면 현장에서 무시하게 된다(경고 피로, alert fatigue). 아래는 각 분석 결과가 이 관점에서 무엇을 뜻하는지 정리한 것이다.

### 1) 경고 물량이 소수 조합에 쏠려 있다 (Q1, Q5)
- 병용금기 사유 문장은 **217종뿐**인데 조합은 23,639개다. 메트포르민+조영제 한 쌍이 제품 조합 기준 **86,404건**으로 압도적이다.
- 현장에서 뜨는 경고의 상당수가 **소수 조합에서 발생**한다는 뜻이다. 이 경고가 반복되면 약사·의사가 습관적으로 닫아버리게 되고, 정작 드물지만 치명적인 금기까지 함께 무시될 위험이 있다.
- **실행 과제**: 메트포르민+조영제는 단순 차단 경고가 아니라 **신기능(eGFR) 수치와 검사 일정에 연동된 조건부 경고**로 바꾸는 것이 효과적이다. "조영제 검사 전 48시간 중단" 같은 실행 지침을 경고문에 포함하면 경고를 닫는 행동 대신 조치를 유도할 수 있다.

### 2) 오래된 금기의 재평가가 필요하다 (Q2)
- 1등급은 평균 9.01년 전 고시로 대부분 오래된 건이고, 최근 고시는 2등급 비중이 높다.
- 오래된 1등급 금기는 **최신 임상 근거가 반영되지 않았을 가능성**이 있다. 실제로는 조건부 사용이 가능한 약이 계속 절대 금기로 차단되면, 환자가 대체 약제로 밀려나 치료 선택지가 줄어든다.
- **실행 과제**: 고시 경과 N년 이상 금기에 대한 **정기 재평가 주기(예: 5년)**를 제도화한다. 이번 분석의 경과기간 계산이 그 대상 목록을 뽑는 기준이 된다.

### 3) 경고 강도를 투여경로별로 차등하는 방안 (Q3)
- 1등급 비율은 내복 19.8%, 주사 10.1%, 외용 5.9%로 명확히 다르다(p=2.2e-18). 바르는 약은 전신 흡수가 적어 위험도가 낮다.
- **제안**: 내복·주사는 차단형 경고, 외용은 정보 제공형 경고로 강도를 나눈다. 외용제까지 같은 강도로 경고하면 경고 피로를 키운다.
- **단, 이 데이터에는 실제 경고가 어떤 형태로 표시되는지에 대한 정보가 없다.** 위 제안은 위험도 차이라는 통계적 근거에서 나온 가설이며, 현행 경고 방식은 DUR 시스템 운영 자료로 따로 확인해야 한다.
- 데이터로 확인되는 사실 하나: 임부금기 1·2등급 사유 문구 1,064종 중 2개 이상 투여경로에 공통으로 쓰인 문구는 83종뿐이다. 즉 **문구 자체는 대체로 경로별로 구분되어 있다**(외용제는 `(외용, 점안제...) 임신 3기에 투여시...`처럼 전용 문구 사용).

### 4) 금기 사유 문구를 표준화할 수 있다 (Q4, Q8)
- '태아'와 '태자', '모체'가 섞여 쓰이고, 같은 내용이 `임부에 대한 안전성 미확립.`과 `임부에 대한 안전성 미확립`(마침표 차이)처럼 여러 버전으로 존재한다. 두 버전을 합치면 464건이며, 사유가 `-`로 비어 있는 건도 57건이다.
- 더 큰 문제는 **같은 문구가 등급을 가리지 못한다**는 점이다. `임부에 대한 안전성 미확립.`은 2등급 263건에 쓰이면서 1등급 16건에도 쓰인다. 문구만 보고는 절대 금기인지 자료 부족인지 알 수 없다.
- 문구가 제각각이면 **검색·집계·시스템 연동이 어렵고**, 제약사·병원이 자체 시스템에 반영할 때 오류가 생긴다.
- LDA 결과 병용금기 사유는 ① 약효 감소 ② 근육·혈관 독성 ③ 이상반응·투여금기 ④ 증후군 ⑤ 심혈관 위험의 5개 유형으로 나뉜다.
- **실행 과제**: 유형별 **표준 문구 템플릿과 표준 용어집**을 만들고, 경고 화면에 위험 유형 아이콘을 붙인다. 현장에서 "무엇이 위험한지"를 읽지 않고도 구분할 수 있게 된다.

### 5) 등급 심사를 자동화해 속도를 높일 수 있다 (Q6, Q7)
- 사유 문장만으로 1등급 여부를 **재현율 89%**로 판별한다. 정형 변수만으로는 0.567에 그치므로, 가치는 텍스트 분석에서 나온다.
- **실행 과제**: 신규 고시 성분의 **등급 초안을 자동 제안**하고 전문가가 검수하는 방식으로 운영한다. 검토 대상을 좁혀 심사 리드타임을 단축할 수 있다.
- 운영 기준: 환자 안전이 걸린 업무이므로 **놓치지 않는 것(재현율)을 우선**하고, 잘못 올린 건은 사람이 걸러내는 구조가 맞다. 7번 모델은 2등급 78건을 1등급으로 잘못 분류했는데, 이는 검수 비용이지 환자 위험은 아니다. 반대로 1등급을 2등급으로 놓치면 곧바로 환자 위험이 된다.
- 한계: 모델은 **기존 등급 부여 관행을 학습**할 뿐 의학적 판단을 대신하지 못한다. 반드시 전문가 검수를 전제로 한 보조 도구로 도입해야 한다.

### 기대 효과 정리
| 과제 | 지표 | 기대 변화 |
|---|---|---|
| 조건부 경고 전환(메트포르민-조영제) | 경고 수용률 | 반복 경고 감소, 실제 조치율 상승 |
| 투여경로별 경고 차등 | 불필요 경고 건수 | 외용제 경고 388건 규모를 정보형으로 전환 |
| 오래된 금기 재평가 | 재평가 대상 수 | 경과 9년 이상 1등급 건 우선 검토 |
| 등급 자동 제안 | 심사 리드타임 | 초안 자동화로 검토 범위 축소 |
| 사유 문구 표준화 | 문구 종류 수 | 217종 → 유형별 템플릿으로 정리 |